In [ ]:
%pip install dlt-meta

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def bronze_custom_transform_func(input_df: DataFrame, dataflow_spec) -> DataFrame:
    """Custom transform that adds column-level comments and a load timestamp.

    Column comments appear in Unity Catalog and enhance data discoverability
    in the Data Intelligence Platform.
    """
    column_comments = {
        "customer_id": "Unique customer identifier",
        "first_name": "Customer first name",
        "last_name": "Customer last name",
        "email": "Customer email address",
        "address": "Customer mailing address",
        "dob": "Customer date of birth",
        "Op": "CDC operation type (I=Insert, U=Update, D=Delete)",
        "dmsTimestamp": "DMS replication timestamp",
    }

    cols = []
    for field in input_df.schema:
        if field.name in column_comments:
            cols.append(
                F.col(field.name).alias(
                    field.name, metadata={"comment": column_comments[field.name]}
                )
            )
        else:
            cols.append(F.col(field.name))

    return input_df.select(*cols).withColumn("load_modified_dt", F.current_timestamp())


def silver_custom_transform_func(input_df: DataFrame, dataflow_spec) -> DataFrame:
    """Custom transform that adds column-level comments to silver table columns."""
    column_comments = {
        "customer_id": "Unique customer identifier",
        "full_name": "Customer full name (first + last)",
        "email": "Customer email address",
    }

    cols = []
    for field in input_df.schema:
        if field.name in column_comments:
            cols.append(
                F.col(field.name).alias(
                    field.name, metadata={"comment": column_comments[field.name]}
                )
            )
        else:
            cols.append(F.col(field.name))

    return input_df.select(*cols)

In [ ]:
layer = spark.conf.get("layer", None)
from src.dataflow_pipeline import DataflowPipeline
DataflowPipeline.invoke_dlt_pipeline(
    spark,
    layer,
    bronze_custom_transform_func=bronze_custom_transform_func,
    silver_custom_transform_func=silver_custom_transform_func,
)